# HW 2 - Разложение матриц градиентным методом

Цель задания: В ходе реализации [разложения Таккера](https://proceedings.neurips.cc/paper/2018/file/45a766fa266ea2ebeb6680fa139d2a3d-Paper.pdf) градиентным методом освоить pyTorch и реализовать подходы оптимизации параметров модели (в отсутствии готовых решений).

[Более-менее внятное описание алгоритма канонического разложения](https://www.alexejgossmann.com/tensor_decomposition_tucker/) - само аналитическое разложение вам реализовывать НЕ НУЖНО

In [19]:
import random
import time
import torch
import pandas as pd
import numpy as np
import tensorly as tl

import scipy.sparse as sparse
from scipy.sparse.linalg import spsolve
from sklearn.preprocessing import MinMaxScaler
from matplotlib import pyplot as plt
from numpy.linalg import svd, matrix_rank, pinv, inv
from scipy.linalg import eigh, eig
from sklearn.metrics import mean_squared_error
from tqdm.notebook import tqdm
from torch import nn

torch.manual_seed(0)

## 1 Создайте 3х мерный тензор
Размер тензора не меньше 100 по каждой из размерностей.

Заполните случайными целыми числами в диапазоне от 0 до 9.

Примечание: разложение будет корректно работать со случайным тензором, только если изначально создавать случайные ядро и матрицы, а потом по ним формировать тензор. Работайте с типом *torch.Tensor.double*.

In [20]:
# Создадим тензор: размер тензора и r задаётся
def get_tensor(size=(100, 200, 150), r=10):
    # data - тензор с заданной размерностью
    # U - список матриц
    # G - ядро разложения

    U = [
        torch.rand(size[i], r, dtype=torch.double, device="cpu") * 2 - 1
        for i in range(3)
    ]
    G = torch.rand(r, r, r, dtype=torch.double, device="cpu") * 2 - 1
    tensor_data = torch.einsum("ijk,ai,bj,ck->abc", G, *U)

    return tensor_data, U, G

Сгенерируйте тензор и добавьте к нему случайный шум с размерностью *1e-2*

In [21]:
size = (100, 200, 300)
r = 10

data, U, G = get_tensor(size, r)
noise = torch.randn_like(data) * 1e-2
data_noisy = data + noise

data_noisy

tensor([[[ 5.2669, -0.8186,  3.7099,  ..., -0.0189, -1.7702,  1.8992],
         [-0.4503,  1.2333, -2.7219,  ...,  3.3630,  1.5060, -1.7975],
         [-2.4275,  0.7831, -0.2424,  ..., -2.1247,  4.5771, -4.1981],
         ...,
         [-3.4374,  1.3801, -2.8872,  ...,  0.2758,  2.2660, -3.7240],
         [-0.0168,  1.2546, -2.6760,  ...,  1.1139, -1.6865,  2.7309],
         [ 6.6482,  2.5793, -2.0965,  ...,  0.4930, -0.1134,  1.5397]],

        [[-0.2553,  1.4461, -1.7238,  ...,  4.2392,  3.8843, -2.0649],
         [-1.9704, -2.0797,  2.1826,  ..., -3.7601, -2.0388,  0.5081],
         [-0.8977, -0.0891, -0.8495,  ...,  0.9351, -2.6140,  1.0751],
         ...,
         [-2.8096, -1.0354,  0.0444,  ..., -0.0434, -1.2846,  1.0435],
         [ 0.2205, -3.0922,  2.9061,  ..., -4.3071, -4.1722,  2.6958],
         [-2.3447,  0.2843, -0.2977,  ...,  1.3555,  2.8080, -1.3936]],

        [[-4.5129, -3.5557,  0.6983,  ...,  0.9755, -0.7961, -0.5252],
         [ 2.7822,  0.3397,  1.7187,  ..., -3

### Вопрос: Почему задание не имеет смысла для полностью случайного тензора и зачем добавлять шум? *не отвечать нельзя*

### Ответ:
#### Для полностью случайного тензора разложение Такера бессмысленно - он не имеет низкоранговой структуры, и разложение не даст сжатия или выявления паттернов.

#### Шум добавляется для:

1. Реализма - реальные данные всегда зашумлены

2. Тестирования устойчивости - проверка, как алгоритм отделяет сигнал от шума

3. Имитации реальных условий - данные редко бывают "идеальными"

#### Таким подходом мы моделируем практическую задачу восстановления полезной информации из зашумленных данных с низкоранговой структурой.

## 2 Реализуйте метод для восстановления тензора по разложению

In [22]:
# Функция, восстанавливающая тензор по ядру и матрицам
def repair_tensor(G_, U):
    # data - восстановленный тензор из матриц и ядра
    # U - список матриц
    # G_ - ядро разложения

    U1, U2, U3 = U

    data = G_.clone()
    data = torch.tensordot(U1, data, dims=([1], [0]))
    data = torch.tensordot(U2, data, dims=([1], [1]))
    data = data.permute(1, 0, 2)

    data = torch.tensordot(U3, data, dims=([1], [2]))
    data = data.permute(1, 2, 0)
    return data

## 3 Сделайте разложение библиотечным методом
Пакет можете брать любой

In [23]:
tl.set_backend("pytorch")


# Восстанавливает тензор из разложения Такера
def reconstruct_from_tucker(G, U):
    return tl.tucker_to_tensor((G, U))


# Считает среднеквадратичную ошибку между тензорами
def calculate_mse(original, reconstructed):
    return torch.mean((original - reconstructed) ** 2)

Не забудьте померить ошибку разложения по метрике MSE

In [24]:
reconstructed_data_our = repair_tensor(G, U)
mse_value_our = calculate_mse(data_noisy, reconstructed_data_our)

print(f"MSE между зашумленным и восстановленным тензором: {mse_value_our.item()}")
print(
    f"MSE между исходным и восстановленным тензором: {calculate_mse(data, reconstructed_data_our).item()}",
    "\n",
)

reconstructed_data_comp = reconstruct_from_tucker(G, U)
mse_value_comp = calculate_mse(data_noisy, reconstructed_data_comp)

print(f"MSE между зашумленным и восстановленным тензором: {mse_value_comp.item()}")
print(
    f"MSE между исходным и восстановленным тензором: {calculate_mse(data, reconstructed_data_comp).item()}"
)

MSE между зашумленным и восстановленным тензором: 0.00010002263705536568
MSE между исходным и восстановленным тензором: 1.7840434703141347e-32 

MSE между зашумленным и восстановленным тензором: 0.00010002263705536568
MSE между исходным и восстановленным тензором: 1.7840434703141347e-32


## 4 Реализуйте разложение градиентным методом

### 4.1 Реализуйте *optimizer*
Можно взять из исходников *PyTorch* и отнаследоваться от *torch.optim.optimizer*.
Используйте квадратичный *Loss*.

In [25]:
import math
import torch
from torch.optim.optimizer import Optimizer


class Opt(Optimizer):
    def __init__(self, params, lr=1e-3):
        config = dict(lr=lr)
        super(Opt, self).__init__(params, config)

    def step(self, closure=None):
        result_loss = None
        if closure is not None:
            result_loss = closure()

        for config_group in self.param_groups:
            step_size = config_group["lr"]

            for weight_tensor in config_group["params"]:
                if weight_tensor.grad is None:
                    continue

                weight_tensor.data = (
                    weight_tensor.data - step_size * weight_tensor.grad.data
                )

        return result_loss

### 4.2 Реализуйте цикл оптимизации параметров

Стоит параметры оптимизировать сразу на GPU

In [26]:
import torch
import torch.nn as nn


# Инициализирует параметры для разложения Такера
def init_params(sizes, rank):
    U = []

    for i in range(3):
        size_i, rank_i = sizes[i], rank
        std = torch.sqrt(torch.tensor(2.0 / (size_i + rank_i)))
        U.append(
            nn.Parameter(
                torch.randn(size_i, rank_i, dtype=torch.double, device="cpu") * std
            )
        )

    G = nn.Parameter(
        torch.randn(rank, rank, rank, dtype=torch.double, device="cpu") * 0.1
    )

    return G, U

In [27]:
size = (100, 200, 150)
r = 10

data, U, G = get_tensor(size, r)
noise = torch.randn_like(data) * 0.01
noisy_data = data + noise

print(f"Исходный тензор: [{data.min().item():.2f}, {data.max().item():.2f}]")
print(f"Зашумленный: [{noisy_data.min().item():.2f}, {noisy_data.max().item():.2f}]")

Исходный тензор: [-24.54, 27.71]
Зашумленный: [-24.55, 27.72]


In [28]:
G_est, U_est = init_params(size, r)
opt = torch.optim.Adam([G_est] + U_est, lr=0.01)

lr_scheduler = torch.optim.lr_scheduler.StepLR(opt, step_size=100, gamma=0.9)

total_epochs = 1000

previous_loss_value = 0.0

for epoch in range(total_epochs):
    opt.zero_grad()
    reconstructed_tensor = repair_tensor(G_est, U_est)
    loss_value = calculate_mse(reconstructed_tensor, noisy_data)
    loss_value.backward()

    torch.nn.utils.clip_grad_norm_([G_est] + U_est, max_norm=1.0)
    opt.step()
    lr_scheduler.step()

    current_loss = loss_value.item()

    if epoch > 0:
        loss_diff = abs(previous_loss_value - current_loss)
        if loss_diff < 1e-10:
            print(f"Остановка на эпохе {epoch}: loss_diff = {loss_diff:.2e}")
            best_core = G_est.detach().clone()
            best_factors = [factor.detach().clone() for factor in U_est]
            break

    previous_loss_value = current_loss
    best_core = G_est.detach().clone()
    best_factors = [factor.detach().clone() for factor in U_est]

    if epoch % 50 == 0:
        learning_rate = opt.param_groups[0]["lr"]
        print(f"Эпоха {epoch}: loss = {current_loss:.8f}, lr = {learning_rate:.6f}")


G_est.data = best_core.data
for idx, factor in enumerate(U_est):
    factor.data = best_factors[idx].data

with torch.no_grad():
    final_reconstruction = repair_tensor(G_est, U_est)
    noisy_mse = calculate_mse(final_reconstruction, noisy_data)
    clean_mse = calculate_mse(final_reconstruction, data)

print(f"MSE относительно зашумленных данных: {noisy_mse.item():.8f}")
print(f"MSE относительно исходных данных: {clean_mse.item():.8f}")

Эпоха 0: loss = 12.17352560, lr = 0.010000
Эпоха 50: loss = 8.21422804, lr = 0.010000
Эпоха 100: loss = 2.24202795, lr = 0.009000
Эпоха 150: loss = 0.01814386, lr = 0.009000
Эпоха 200: loss = 0.00021023, lr = 0.008100
Эпоха 250: loss = 0.00010037, lr = 0.008100
Эпоха 300: loss = 0.00009975, lr = 0.007290
Остановка на эпохе 312: loss_diff = 9.37e-11
MSE относительно зашумленных данных: 0.00009975
MSE относительно исходных данных: 0.00000018


## 5 Приведите сравнение скорости работы и ошибки восстановления методом из пакета и реализованного градиентного
Сравнение может считаться ± объективным с размером выборки от 10.

In [ ]:
def compare_methods(num_experiments=10):
    tl.set_backend("numpy")

    gradient_times = []
    gradient_errors = []

    library_times = []
    library_errors = []

    for experiment_idx in range(num_experiments):
        print(f"Эксперимент № {experiment_idx + 1}")

        dimensions = (100, 200, 150)
        rank_value = 10
        original_tensor, true_factors, true_core = get_tensor(dimensions, rank_value)
        noise_component = torch.randn_like(original_tensor) * 1e-2
        noisy_tensor = original_tensor + noise_component

        # Градиентный метод

        timer = time.time()

        G_est, U_est = init_params(dimensions, rank_value)
        opt = torch.optim.Adam([G_est] + U_est, lr=0.01)
        lr_scheduler = torch.optim.lr_scheduler.StepLR(opt, step_size=100, gamma=0.9)

        max_epochs = 1000
        previous_loss = 0.0
        best_G = None
        best_U = None

        for epoch_idx in range(max_epochs):
            opt.zero_grad()
            repaired = repair_tensor(G_est, U_est)
            current_loss = calculate_mse(repaired, noisy_tensor)
            current_loss.backward()

            torch.nn.utils.clip_grad_norm_([G_est] + U_est, max_norm=1.0)
            opt.step()
            lr_scheduler.step()

            loss_value = current_loss.item()

            if epoch_idx > 0:
                loss_diff = abs(previous_loss - loss_value)

                if loss_diff < 1e-10:
                    print(
                        f"Остановка на эпохе {epoch_idx}: loss_diff = {loss_diff:.2e}"
                    )
                    best_G = G_est.detach().clone()
                    best_U = [u.detach().clone() for u in U_est]
                    break

            previous_loss = loss_value
            best_G = G_est.detach().clone()
            best_U = [u.detach().clone() for u in U_est]

        if best_G is not None:
            G_est.data = best_G.data
            for i, j in enumerate(U_est):
                j.data = best_U[i].data

        with torch.no_grad():
            gradient_repaired = repair_tensor(G_est, U_est)
            gradient_error = calculate_mse(gradient_repaired, original_tensor).item()

        gradient_duration = time.time() - timer
        gradient_times.append(gradient_duration)
        gradient_errors.append(gradient_error)
        print(
            f"Градиентный метод: время = {gradient_duration:.2f} с., MSE = {gradient_error:.8f}"
        )

        # Метод из библиотеки

        timer = time.time()

        numpy_tensor = noisy_tensor.detach().numpy()

        core_lib, factors_lib = tl.decomposition.tucker(
            numpy_tensor, rank=[rank_value, rank_value, rank_value]
        )

        library_reconstruction = tl.tucker_to_tensor((core_lib, factors_lib))
        library_reconstruction = torch.from_numpy(library_reconstruction).double()

        library_duration = time.time() - timer
        library_times.append(library_duration)
        library_error = calculate_mse(library_reconstruction, original_tensor).item()
        library_errors.append(library_error)
        print(
            f"  Библиотечный метод: время = {library_duration:.2f} с., MSE = {library_error:.8f}"
        )
        print()

    print("=" * 60)
    print("ИТОГИ:")
    print("=" * 60)

    print()

    print("Градиентный метод:")
    print(
        f"  Среднее время выполнения: {np.mean(gradient_times):.2f} ± {np.std(gradient_times):.2f} с"
    )
    print(
        f"  Средняя величина ошибки: {np.mean(gradient_errors):.2e} ± {np.std(gradient_errors):.2e}"
    )

    print()

    print("Библиотечный метод:")
    print(
        f"  Среднее время выполнения: {np.mean(library_times):.2f} ± {np.std(library_times):.2f} с."
    )
    print(
        f"  Средняя величина ошибки: {np.mean(library_errors):.2e} ± {np.std(library_errors):.2e}"
    )
    print()


comparison_results = compare_methods(num_experiments=10)

Эксперимент № 1
Остановка на эпохе 342: loss_diff = 9.66e-11
Градиентный метод: время = 20.30 с., MSE = 0.00000017
  Библиотечный метод: время = 1.89 с., MSE = 0.00000017

Эксперимент № 2
Остановка на эпохе 306: loss_diff = 9.08e-11
Градиентный метод: время = 19.93 с., MSE = 0.00000018
  Библиотечный метод: время = 1.98 с., MSE = 0.00000018

Эксперимент № 3
Остановка на эпохе 318: loss_diff = 9.71e-11
Градиентный метод: время = 18.07 с., MSE = 0.00000018
  Библиотечный метод: время = 1.78 с., MSE = 0.00000017

Эксперимент № 4
Остановка на эпохе 310: loss_diff = 9.24e-11
Градиентный метод: время = 21.16 с., MSE = 0.00000018
  Библиотечный метод: время = 1.96 с., MSE = 0.00000018

Эксперимент № 5
Остановка на эпохе 351: loss_diff = 9.86e-11
Градиентный метод: время = 23.58 с., MSE = 0.00000017
  Библиотечный метод: время = 5.12 с., MSE = 0.00000017

Эксперимент № 6
Остановка на эпохе 314: loss_diff = 9.15e-11
Градиентный метод: время = 19.26 с., MSE = 0.00000018
  Библиотечный метод: вре